# Phục hồi trace SymPL cũ

Chạy trong session còn thư mục output cũ, hoặc mount output và sửa OLD_OUTPUT. Không tải model hay chạy inference.


In [ ]:
REPO_URL = 'https://github.com/leetranwyng/sc203-kat.git'
CODE_REF = 'cc722ce3ba68264e34dd7c640ecd6029009629b1'
HF_SECRET_NAME = 'HF_token'
RUN_CONFIG = {'method': 'sympl', 'mode': 'sanity', 'trace_input': None, 'model_input': None, 'resume_input': None, 'run_smoke_first': True, 'runtime_dir': '/kaggle/temp/sc203-kat/sympl_trace', 'output_dir': '/kaggle/working/sc203-kat/sympl_trace_recover', 'method_options': {'category_policy': 'dataset_mapping', 'max_new_tokens': 1024, 'max_pixels': None, 'quantization': None, 'retry_errors': False}}
OLD_OUTPUT = '/kaggle/working/sympl_trace120_sanity_v2'


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

if not CODE_REF or CODE_REF.startswith('-') or any(c.isspace() for c in CODE_REF):
    raise ValueError('CODE_REF must be a Git tag, branch, or full commit.')
if REPO_URL != 'https://github.com/leetranwyng/sc203-kat.git':
    raise ValueError('Use the public leetranwyng/sc203-kat repository.')
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT='0', GCM_INTERACTIVE='Never')
cache = Path(os.environ.get('SC203_CODE_CACHE', '/kaggle/temp/sc203-kat-code'))
cache.mkdir(parents=True, exist_ok=True)
key = hashlib.sha256((REPO_URL + '\n' + CODE_REF).encode()).hexdigest()[:16]
CODE_ROOT = cache / key

def git(*args):
    result = subprocess.run(['git', *args], env=GIT_ENV, text=True,
                            capture_output=True, timeout=300)
    if result.returncode: raise RuntimeError((result.stderr or result.stdout).strip())
    return result.stdout.strip()

if not (CODE_ROOT / '.git').exists():
    if CODE_ROOT.exists() and any(CODE_ROOT.iterdir()):
        raise RuntimeError('Code cache is not empty; choose a different SC203_CODE_CACHE.')
    git('clone', '--no-checkout', '--no-tags', '--depth', '1', REPO_URL, str(CODE_ROOT))
else:
    if git('-C', str(CODE_ROOT), 'remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('Cached source origin differs from REPO_URL.')
    if git('-C', str(CODE_ROOT), 'status', '--porcelain', '--untracked-files=normal'):
        raise RuntimeError('Cached source has local changes; use a new code cache.')
git('-C', str(CODE_ROOT), 'fetch', '--depth', '1', '--no-tags', 'origin', CODE_REF)
commit = git('-C', str(CODE_ROOT), 'rev-parse', '--verify', 'FETCH_HEAD^{commit}')
git('-C', str(CODE_ROOT), 'checkout', '--detach', commit)

ENV = dict(GIT_ENV, SC203_KAT_ROOT=str(CODE_ROOT), PYTHONPATH=str(CODE_ROOT / 'src'))
if HF_SECRET_NAME and not ENV.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(HF_SECRET_NAME)
        if value: ENV['HF_TOKEN'] = value.strip()
        del value
    except Exception:
        print('HF secret unavailable; using public/local model files.')
config_dir = Path(os.environ.get('SC203_CONFIG_DIR', '/kaggle/temp/sc203-kat-configs'))
config_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = config_dir / (RUN_CONFIG['method'] + '_' + RUN_CONFIG['mode'] + '.json')
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
subprocess.run([sys.executable, '-m', 'sc203_kat.sympl_trace', '--recover-output', OLD_OUTPUT], cwd=CODE_ROOT, env=ENV, check=True)


In [ ]:
from IPython.display import display, FileLink
output = Path(OLD_OUTPUT)
for p in sorted(output.rglob('sanity_report.json')): print(p, p.read_text())
for p in sorted(output.parent.joinpath(output.name+'_downloads').glob('*.zip')): display(FileLink(str(p)))
